# A reported portfolio, not investment performance

This notebook starts with invented positions and needs only Python's standard library.
Run Jupyter from the repository root (`jupyter notebook notebooks/reported_portfolio.ipynb`).
Jupyter is optional and installed separately. Committed outputs are intentionally empty.

We compute top-position concentration using known reported dollar values. This does not
estimate returns, exposure outside the selected table, or a person's personal wealth.


In [ ]:
from pathlib import Path
from decimal import Decimal
import json
import sys

ROOT = Path.cwd()
if not (ROOT / "fixtures").is_dir():
    ROOT = ROOT.parent
if not (ROOT / "fixtures" / "portfolio.synthetic.json").is_file():
    raise RuntimeError("Run from the repository root or notebooks directory")
with (ROOT / "fixtures" / "portfolio.synthetic.json").open() as stream:
    data = json.load(stream, parse_float=Decimal)
print("SYNTHETIC DATA", data["quarter"], data["period"])


## Optional live data: exactly one call

Leave `LIVE = False` for offline use. Before setting it to true, read
https://disclosery.com/data and set `DISCLOSERY_API_KEY` only in your local environment
if desired. CIK 1350694 identifies Bridgewater; inspect the returned reporter and citations.
No fixed quarter is assumed. This selects at most ten rows from the latest available
quarter and does not fetch further pages. Errors stop execution.


In [ ]:
LIVE = False
if LIVE:
    sys.path.insert(0, str(ROOT / "api"))
    from client import get
    envelope = get("/api/v1/funds/1350694/portfolio", view="summary", limit=10, sort="value")
    data = envelope["data"]
    print("Quota:", envelope["quota"])
    print("Reporter:", data.get("filer"))
    print("Period:", data.get("quarter"), data.get("period"))
    print("Sources:", data.get("filings"), data.get("source_url"), data.get("edgar_url"))
    print("Summary and warnings:", data.get("summary"), "no_base:", data.get("no_base"))


## Display coverage before arithmetic

`summary.value_total_usd` is the denominator for eligible share-position ratios, not
necessarily the cover total or total assets under management. Unknown values remain
unknown. Complete imported coverage does not make 13F a complete investment portfolio.


In [ ]:
summary = data.get("summary", {})
print("Paging:", data.get("paging"))
print("Complete flag:", summary.get("complete"))
print("Notes:", summary.get("notes", []))
rows = data.get("holdings", [])
denominator = summary.get("value_total_usd")
known = [row for row in rows if row.get("value_usd") is not None]
known.sort(key=lambda row: Decimal(str(row["value_usd"])), reverse=True)
for row in known:
    print(row.get("name"), "USD", row["value_usd"], "reported %", row.get("pct_portfolio"), "notes", row.get("notes", []))
if denominator is None or Decimal(str(denominator)) <= 0:
    print("No usable denominator; concentration is unavailable.")
else:
    # Preserve the API's eligibility rules by summing supplied row percentages.
    # Recomputing value / total for flagged rows could produce misleading ratios.
    weights = [Decimal(str(row["pct_portfolio"])) for row in known[:3] if row.get("pct_portfolio") is not None]
    if len(weights) != min(3, len(known)):
        print("Some top-row ratios unavailable; no combined concentration reported.")
    elif weights:
        print("Top", len(weights), "displayed known-value positions: reported concentration", sum(weights), "%")
    else:
        print("No known-value positions.")


## Interpretation checklist

- Is the reporting organization the one you intended?
- Which quarter and filing/amendment establish these reported facts?
- Are pages, tables or local coverage missing? Do not silently fill gaps with zeroes.
- Preserve the supplied ratios and warnings; options and principal amounts are separate.
- Cite the returned Disclosery and SEC URLs. Do not infer performance from a snapshot.

For live data, clear outputs before committing or sharing. This example deliberately
avoids bulk identifier mappings and exports.
